In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# Qwen2.5-VL LoRA fine-tune (optional)

Minimal LoRA scaffold. You may need to adjust the prompt and masking for your model.


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from transformers import AutoProcessor, AutoModelForCausalLM, Trainer, TrainingArguments
try:
    from transformers import AutoModelForVision2Seq
except Exception:
    AutoModelForVision2Seq = None

try:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    HAS_PEFT = True
except Exception:
    HAS_PEFT = False


2026-02-03 11:58:37.693064: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-03 11:58:37.693103: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-03 11:58:37.694043: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-03 11:58:37.699880: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-03 11:58:38.562601: W tensorflow/compiler/tf2

In [3]:
from pathlib import Path
import os

def find_imageclef_root() -> Path:
    env_root = os.environ.get("IMAGECLEF_MEDVQA_GI_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"IMAGECLEF_MEDVQA_GI_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "ImageCLEF_MEDVQA_GI_2023" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "ImageCLEF_MEDVQA_GI_2023" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate ImageCLEF_MEDVQA_GI_2023 root. "
        "Run from within the ImageCLEF_MEDVQA_GI_2023 folder or set IMAGECLEF_MEDVQA_GI_ROOT."
    )

ROOT = find_imageclef_root()
sys.path.append(str(ROOT))

from common import (
    find_long_table,
    load_long_table,
    normalize_answer,
)

DATA_PATH = find_long_table(ROOT)
OUT_DIR = ROOT / "3_modern_vlm" / "out" / "06_qwen2_5_vl_lora_finetune"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
BATCH_SIZE = 2
EPOCHS = 1
LR = 2e-4
MAX_TRAIN_SAMPLES = int(os.environ.get("MAX_TRAIN_SAMPLES", "0")) or None
MAX_TEXT_LEN = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [4]:
if not HAS_PEFT:
    raise ImportError("peft is required for LoRA fine-tuning. Install with `pip install peft`. ")

long_df = load_long_table(DATA_PATH)
train_df = long_df[long_df["split"] == "train"].copy()
if MAX_TRAIN_SAMPLES:
    train_df = train_df.head(MAX_TRAIN_SAMPLES)


In [5]:
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)

if AutoModelForVision2Seq is not None:
    try:
        base_model = AutoModelForVision2Seq.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
            trust_remote_code=True,
        )
    except Exception:
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
            trust_remote_code=True,
        )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        trust_remote_code=True,
    )

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
model.to(DEVICE)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

ValueError: Please specify `target_modules` or `target_parameters`in `peft_config`

In [ ]:
# Prompt helpers

def format_prompt(question: str) -> str:
    q = question.strip()
    if hasattr(processor, "apply_chat_template"):
        conversation = [
            {"role": "system", "content": "You are a concise medical VQA assistant."},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": q}]},
        ]
        return processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    return f"USER: <image>Question: {q}ASSISTANT:"


In [ ]:
class VQALoRADataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        question = row["question_text"]
        answer = normalize_answer(row["answer_norm"])

        prompt = format_prompt(question)
        full_text = prompt + " " + answer

        full = processor(images=image, text=full_text, return_tensors="pt", padding="max_length", truncation=True, max_length=MAX_TEXT_LEN)
        prompt_only = processor(images=image, text=prompt, return_tensors="pt", padding="max_length", truncation=True, max_length=MAX_TEXT_LEN)

        input_ids = full["input_ids"].squeeze(0)
        attention_mask = full["attention_mask"].squeeze(0)
        pixel_values = full.get("pixel_values", None)
        if pixel_values is not None:
            pixel_values = pixel_values.squeeze(0)

        labels = input_ids.clone()
        prompt_len = prompt_only["input_ids"].shape[1]
        labels[:prompt_len] = -100

        item = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }
        if pixel_values is not None:
            item["pixel_values"] = pixel_values
        return item

train_ds = VQALoRADataset(train_df)


In [ ]:
args = TrainingArguments(
    output_dir=str(OUT_DIR / "checkpoints"),
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_strategy="epoch",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
)


In [ ]:
# Train
trainer.train()


In [ ]:
# Save LoRA adapter
model.save_pretrained(OUT_DIR / "lora_adapter")
OUT_DIR
